# T16 — Consolidação dos 20 experimentos

Esta tabela lê **os resultados incorporados aos notebooks**; não cria arquivos de resultados. Um MAE só entra no ranking final quando os quatro modelos da mesma base tiverem previsões nas mesmas origens do protocolo T08. MAEs de bases diferentes têm unidades e escalas diferentes e não são somados.

**Estado atual:** os cinco Random Forest têm resultados executados. O SARIMAX do grupo 4, embora executado na `main`, tratou a fonte por uma rota diferente da base comum e fica fora do ranking até a integração. O Holt-Winters do grupo 1 está mensal e também fica fora do protocolo diário.

In [1]:
from pathlib import Path
import json
import re
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'preparacao_bases.py').is_file())
BASES = {1:'bitcoin', 2:'trafego', 3:'poluicao', 4:'clima', 5:'ouro'}
MODELOS = ('Random Forest','Holt-Winters','SARIMAX','MLP Regressor')
MANIFESTO = json.loads((RAIZ/'dados_tratados'/'manifesto.json').read_text(encoding='utf-8'))['bases']

## Leitura dos resultados executados

O extrator exige MAE e quantidade de previsões no output de cada notebook RF. Se alguma célula for limpa ou o formato mudar, ele sinaliza a ausência; não reutiliza um número antigo silenciosamente.

In [2]:
def textos_saida(caminho):
    notebook = nbformat.read(caminho, as_version=4)
    partes = []
    for celula in notebook.cells:
        for saida in celula.get('outputs', []):
            partes.append(''.join(saida.get('text', [])))
            partes.append(''.join(saida.get('data', {}).get('text/plain', [])))
    return '\n'.join(partes)

def ler_rf(grupo):
    caminho = RAIZ/f'grupo{grupo}'/f'random-forest-{grupo}.ipynb'
    texto = textos_saida(caminho)
    if grupo == 4:
        achado = re.search(r'^\s*0\s+Random Forest\s+([\d,]+)\s+([\d.]+)', texto, re.M)
        if not achado:
            raise ValueError(f'{caminho}: tabela final do RF não encontrada')
        n, mae = int(achado.group(1).replace(',', '')), float(achado.group(2))
        base_encontrada = re.search(r'^\s*1\s+Persistência \(1h\)\s+[\d,]+\s+([\d.]+)', texto, re.M)
        if not base_encontrada: raise ValueError(f'{caminho}: MAE da persistência não encontrado')
        mae_persistencia = float(base_encontrada.group(1))
    else:
        mae_encontrado = re.search(r'MAE RF:\s*([\d.]+)', texto)
        n_encontrado = re.search(r'previsões:\s*([\d,]+)', texto)
        if not mae_encontrado or not n_encontrado:
            raise ValueError(f'{caminho}: execute T11/T12/T16 antes da consolidação')
        mae = float(mae_encontrado.group(1))
        n = int(n_encontrado.group(1).replace(',', ''))
        base_encontrada = re.search(r'MAE persistência:\s*([\d.]+)', texto)
        if not base_encontrada: raise ValueError(f'{caminho}: MAE da persistência não encontrado')
        mae_persistencia = float(base_encontrada.group(1))
    meta = MANIFESTO[BASES[grupo]]
    return {'grupo':grupo, 'base':BASES[grupo], 'modelo':'Random Forest',
            'MAE':mae, 'MAE_persistencia':mae_persistencia,
            'ganho_MAE_vs_persistencia_%':100*(1-mae/mae_persistencia),
            'n':n, 'cobertura_grade_%':100*n/meta['linhas_teste'],
            'origem':str(caminho.relative_to(RAIZ))}

rf = pd.DataFrame([ler_rf(g) for g in BASES])
display(rf.round(4))

,grupo,base,modelo,MAE,MAE_persistencia,ganho_MAE_vs_persistencia_%,n,cobertura_grade_%,origem
0,1,bitcoin,Random Forest,1661.4218,1386.9603,-19.7887,884,100.0000,grupo1\random-forest-1.ipynb
1,2,trafego,Random Forest,153.2877,590.7910,74.0538,10267,97.6786,grupo2\random-forest-2.ipynb
2,3,poluicao,Random Forest,9.8670,10.3906,5.0392,6336,90.3465,grupo3\random-forest-3.ipynb
3,4,clima,Random Forest,0.3951,0.6865,42.4472,13785,98.2818,grupo4\random-forest-4.ipynb
4,5,ouro,Random Forest,36.1629,19.5735,-84.7544,601,100.0000,grupo5\random-forest-5.ipynb


## Matriz de entregas e ranking

As células dos demais modelos ficam explicitamente pendentes. Para incorporá-los, seus notebooks precisam exibir MAE, número de previsões e datas de teste. A comparação final exige checar essas datas com `validar_previsoes` no notebook de cada modelo.

In [3]:
grade = pd.MultiIndex.from_product([BASES, MODELOS], names=['grupo','modelo']).to_frame(index=False)
grade['base'] = grade['grupo'].map(BASES)
consolidado = grade.merge(rf[['grupo','modelo','MAE','n','cobertura_grade_%','origem']],
                          on=['grupo','modelo'], how='left')
consolidado['situacao'] = np.where(consolidado['MAE'].notna(), 'resultado comparável', 'pendente')
observacoes = {
    (1, 'Holt-Winters'): 'protótipo mensal; adaptar ao diário',
    (4, 'SARIMAX'): 'executado com tratamento próprio; alinhar e refazer',
    (4, 'MLP Regressor'): 'notebook sem execução incorporada',
}
for (grupo, modelo), situacao in observacoes.items():
    consolidado.loc[(consolidado.grupo == grupo) & (consolidado.modelo == modelo), 'situacao'] = situacao
display(consolidado.sort_values(['grupo','modelo']).round(4))
print(f"Resultados elegíveis hoje: {consolidado['MAE'].notna().sum()}/20")

completos = consolidado.groupby('grupo').filter(lambda p: p['MAE'].notna().all())
if completos.empty:
    print('Ranking, vitórias e posição média pendentes: nenhuma base tem os quatro modelos comparáveis.')
else:
    completos = completos.copy()
    completos['posicao'] = completos.groupby('grupo')['MAE'].rank(method='min')
    ranking = completos.groupby('modelo').agg(
        vitorias=('posicao', lambda p: int((p == 1).sum())),
        posicao_media=('posicao','mean'), bases=('grupo','nunique')
    ).sort_values(['vitorias','posicao_media'], ascending=[False,True])
    display(ranking)


,grupo,modelo,base,MAE,n,cobertura_grade_%,origem,situacao
1,1,Holt-Winters,bitcoin,NaN,NaN,NaN,NaN,protótipo mensal; adaptar ao diário
3,1,MLP Regressor,bitcoin,NaN,NaN,NaN,NaN,pendente
0,1,Random Forest,bitcoin,1661.4218,884.0,100.0000,grupo1\random-forest-1.ipynb,resultado comparável
2,1,SARIMAX,bitcoin,NaN,NaN,NaN,NaN,pendente
5,2,Holt-Winters,trafego,NaN,NaN,NaN,NaN,pendente
7,2,MLP Regressor,trafego,NaN,NaN,NaN,NaN,pendente
4,2,Random Forest,trafego,153.2877,10267.0,97.6786,grupo2\random-forest-2.ipynb,resultado comparável
6,2,SARIMAX,trafego,NaN,NaN,NaN,NaN,pendente
9,3,Holt-Winters,poluicao,NaN,NaN,NaN,NaN,pendente
11,3,MLP Regressor,poluicao,NaN,NaN,NaN,NaN,pendente


Resultados elegíveis hoje: 5/20
Ranking, vitórias e posição média pendentes: nenhuma base tem os quatro modelos comparáveis.


### Antes do relatório final

1. Alinhar as bases, origens e horizontes dos quatro modelos em cada grupo.
2. Executar e incorporar os 15 pipelines ainda ausentes desta matriz.
3. Conferir cobertura e datas antes de gerar ranking e número de vitórias.
4. Interpretar resultados e limitações por base, sem comparar MAEs brutos entre unidades diferentes.